In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score, accuracy_score)

import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

print("Librerías importadas correctamente.")

## Explicación del dataset

In [ ]:
data = pd.read_csv("Pokemon.csv")
data.head()

# Explicación del Dataset

El dataset contiene información de 800 Pokémon con sus estadísticas base. Cada fila representa un Pokémon y cada columna una característica específica.

La variable objetivo es `Legendary`, lo que convierte esto en un **problema de clasificación binaria**.

---

## Descripción de las columnas

| Columna | Tipo de dato | Descripción |
|---|---|---|
| `#` | Entero | Número identificador en la Pokédex. No es informativo para el modelo (puede repetirse en mega-evoluciones). |
| `Name` | Texto | Nombre del Pokémon. No es informativo numéricamente. |
| `Type 1` | Texto | Tipo principal del Pokémon (18 tipos posibles). Variable categórica. |
| `Type 2` | Texto / Nulo | Tipo secundario. 386 Pokémon no tienen segundo tipo (valor nulo). Variable categórica. |
| `Total` | Entero | Suma de todas las estadísticas base. Variable redundante si se incluyen las individuales, pero puede ser útil. |
| `HP` | Entero | Puntos de vida (Hit Points). |
| `Attack` | Entero | Ataque físico. |
| `Defense` | Entero | Defensa física. |
| `Sp. Atk` | Entero | Ataque especial. |
| `Sp. Def` | Entero | Defensa especial. |
| `Speed` | Entero | Velocidad. |
| `Generation` | Entero | Generación del Pokémon (1–6). Variable ordinal. |
| `Legendary` | Booleano | **Variable objetivo**: True si el Pokémon es legendario, False en caso contrario. |

---

## Información relevante

- **Desbalance de clases**: 735 no-legendarios vs 65 legendarios (~8% de la clase positiva). Esto requiere métricas apropiadas (F1, AUC-ROC) y particiones estratificadas.
- **Valores nulos**: Solo `Type 2` tiene nulos (386 instancias), que serán tratados como una categoría propia ("None").
- **Variables categóricas**: `Type 1` y `Type 2` deben codificarse numéricamente (one-hot encoding).
- **Variables no informativas**: `#` y `Name` se eliminan al no aportar información predictiva generalizable.

## Preparación del dataset

In [ ]:
# ─────────────────────────────────────────────────────────
# 1. Eliminar columnas no informativas
# Justificación: '#' puede repetirse (mega-evoluciones) y 'Name' es texto libre,
# ambas son identificadores sin capacidad predictiva generalizable.
# ─────────────────────────────────────────────────────────
df = data.copy()
df = df.drop(columns=['#', 'Name'])

# ─────────────────────────────────────────────────────────
# 2. Tratar valores nulos en Type 2
# Justificación: 386 Pokémon no tienen segundo tipo. Reemplazamos NaN con 'None'
# (categoría propia) para no perder esas instancias y para que el modelo pueda
# aprender que "no tener segundo tipo" es una característica relevante.
# ─────────────────────────────────────────────────────────
df['Type 2'] = df['Type 2'].fillna('None')

# ─────────────────────────────────────────────────────────
# 3. One-hot encoding de variables categóricas
# Justificación: Las redes neuronales requieren entradas numéricas. El one-hot
# encoding evita introducir un orden artificial entre los tipos (ej. Grass=1,
# Fire=2 implicaría que Fire > Grass, lo cual no tiene sentido).
# ─────────────────────────────────────────────────────────
df = pd.get_dummies(df, columns=['Type 1', 'Type 2'])

# ─────────────────────────────────────────────────────────
# 4. Convertir variable objetivo a entero
# ─────────────────────────────────────────────────────────
df['Legendary'] = df['Legendary'].astype(int)

print("Shape tras preprocesamiento:", df.shape)
print("\nColumnas:")
print(list(df.columns))
print("\nDistribución de clases:")
print(df['Legendary'].value_counts())

In [ ]:
# ─────────────────────────────────────────────────────────
# 5. Separar features y target
# ─────────────────────────────────────────────────────────
X = df.drop(columns=['Legendary']).values.astype(np.float32)
y = df['Legendary'].values.astype(np.float32)

input_dim = X.shape[1]
print(f"Dimensión de entrada: {input_dim} features")

# ─────────────────────────────────────────────────────────
# 6. División train / val / test con estratificación
# Justificación: La estratificación garantiza que la proporción de legendarios
# se mantenga en cada split, crítico con solo 65 ejemplos positivos.
# Usamos 70% train, 15% val, 15% test.
# ─────────────────────────────────────────────────────────
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15/(1-0.15), stratify=y_trainval, random_state=42)

print(f"Train: {X_train.shape[0]} ({int(y_train.sum())} legendarios)")
print(f"Val:   {X_val.shape[0]}  ({int(y_val.sum())} legendarios)")
print(f"Test:  {X_test.shape[0]} ({int(y_test.sum())} legendarios)")

In [ ]:
# ─────────────────────────────────────────────────────────
# 7. Normalización (StandardScaler)
# Justificación: Las estadísticas de los Pokémon tienen rangos distintos
# (HP: 1-255, Total: 180-780). La normalización z-score lleva todos los
# features a media 0 y desviación estándar 1, lo que estabiliza el 
# gradiente descendente y mejora la convergencia del MLP, como indica
# el material del curso (Glorot y Bengio, 2010).
# IMPORTANTE: fit solo en train set para evitar data leakage.
# ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# ─────────────────────────────────────────────────────────
# 8. Convertir a tensores PyTorch y crear DataLoaders
# ─────────────────────────────────────────────────────────
def make_loader(X, y, batch_size=32, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 32
train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

print(f"DataLoaders creados con batch_size={BATCH_SIZE}")

## Definición del modelo

In [ ]:
from models import MLP_Simple, MLP_Medium, MLP_Deep

# Instanciar las tres arquitecturas
model_simple = MLP_Simple(input_dim)
model_medium = MLP_Medium(input_dim)
model_deep   = MLP_Deep(input_dim)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Arquitecturas definidas:")
print(f"  MLP_Simple : {count_params(model_simple):,} parámetros")
print(f"  MLP_Medium : {count_params(model_medium):,} parámetros")
print(f"  MLP_Deep   : {count_params(model_deep):,} parámetros")
print()
print("Arquitectura seleccionada: MLP_Medium")
print(model_medium)

### Justificación de la arquitectura seleccionada: MLP_Medium

Se selecciona **MLP_Medium** (2 capas ocultas + Dropout) como la mejor arquitectura por las siguientes razones:

1. **Capacidad adecuada**: Con 64 → 32 neuronas ocultas, tiene suficiente capacidad para aprender las relaciones no lineales entre estadísticas y condición legendaria, sin ser excesivamente compleja para un dataset de 800 instancias.

2. **Regularización con Dropout(0.3)**: El dataset está fuertemente desbalanceado (8% legendarios) y es pequeño. El Dropout reduce el sobreajuste al apagar aleatoriamente neuronas durante el entrenamiento.

3. **Activaciones ReLU**: Evitan el problema del gradiente desvaneciente presente en sigmoid/tanh en capas ocultas, permitiendo un entrenamiento más estable.

4. **Comparación con alternativas**:
   - `MLP_Simple`: Una sola capa puede ser insuficiente para capturar interacciones entre los 45 features.
   - `MLP_Deep`: Con BatchNorm y 3 capas, es más compleja de lo necesario para este dataset pequeño y tiene mayor riesgo de sobreajuste.

## Definición de optimizador y función de costo

In [ ]:
# ─────────────────────────────────────────────────────────
# Función de costo: BCELoss (Binary Cross-Entropy)
# Justificación: Es la función de costo estándar para clasificación binaria.
# Como el modelo usa Sigmoid en la salida, la salida ya está en [0,1]
# y puede interpretarse como probabilidad. BCELoss mide la divergencia
# entre las probabilidades predichas y las etiquetas reales.
#
# Manejo del desbalance: Se usa pos_weight para dar más importancia
# a los ejemplos de la clase minoritaria (legendarios). El peso se
# calcula como n_negativos / n_positivos ≈ 735/65 ≈ 11.3
# ─────────────────────────────────────────────────────────
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f"pos_weight: {pos_weight.item():.2f}  (n_neg={n_neg}, n_pos={n_pos})")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Nota: Usamos BCEWithLogitsLoss (más numericamente estable) en lugar de
# BCELoss, por lo que debemos quitar el Sigmoid final del modelo de entrenamiento.
# Redefinimos el modelo sin Sigmoid para el entrenamiento (se aplica internamente):
class MLP_Medium_Logits(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)  # Sin Sigmoid, BCEWithLogitsLoss lo incluye internamente
        )
    def forward(self, x):
        return self.model(x)

model = MLP_Medium_Logits(input_dim)

# ─────────────────────────────────────────────────────────
# Optimizador: Adam con lr=1e-3
# Justificación: Adam (Kingma y Ba, 2014) adapta la tasa de aprendizaje
# individualmente para cada parámetro usando estimaciones de primer y segundo
# momento del gradiente. Es el estándar de facto para entrenar MLPs y converge
# más rápido que SGD simple. Se usa weight_decay=1e-4 como regularización L2
# adicional al Dropout.
# ─────────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ─────────────────────────────────────────────────────────
# Learning rate scheduler: ReduceLROnPlateau
# Justificación: Implementa el concepto de "annealing" visto en clases.
# Reduce la tasa de aprendizaje a la mitad si la pérdida de validación
# no mejora en 10 épocas consecutivas, permitiendo convergencia más fina.
# ─────────────────────────────────────────────────────────
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True)

print("\nOptimizador:", optimizer)
print("\nFunción de costo: BCEWithLogitsLoss con pos_weight =", pos_weight.item())

## Entrenamiento del modelo

In [ ]:
# ─────────────────────────────────────────────────────────
# Función de entrenamiento por época
# ─────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        logits = model(X_batch).squeeze(1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    return total_loss / len(loader.dataset)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch).squeeze(1)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            preds = torch.sigmoid(logits) >= 0.5
            all_preds.extend(preds.numpy())
            all_labels.extend(y_batch.numpy())
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, f1

# ─────────────────────────────────────────────────────────
# Hiperparámetros de entrenamiento
# - EPOCHS=150: suficiente para convergencia con early stopping
# - PATIENCE=20: número de épocas sin mejora antes de detener
# Justificación: Con early stopping evitamos sobreajuste y ahorramos cómputo.
# El modelo se guarda cuando alcanza el mejor F1 de validación.
# ─────────────────────────────────────────────────────────
EPOCHS   = 150
PATIENCE = 20

train_losses, val_losses = [], []
train_f1s,    val_f1s    = [], []

best_val_f1   = -1
best_weights  = None
patience_cnt  = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss          = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1 = eval_epoch(model, val_loader, criterion)
    _, tr_f1         = eval_epoch(model, train_loader, criterion)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    train_f1s.append(tr_f1)
    val_f1s.append(val_f1)

    scheduler.step(val_loss)

    # Checkpointing: guardar el mejor modelo en validación
    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f"Época {epoch:3d} | Loss Train: {tr_loss:.4f} Val: {val_loss:.4f} "
              f"| F1 Train: {tr_f1:.3f} Val: {val_f1:.3f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if patience_cnt >= PATIENCE:
        print(f"\nEarly stopping en época {epoch} (mejor F1-val: {best_val_f1:.4f})")
        break

# Restaurar los mejores pesos
model.load_state_dict(best_weights)
print(f"\nEntrenamiento finalizado. Mejor F1 en validación: {best_val_f1:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────
# Curvas de aprendizaje
# ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='darkorange')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('BCEWithLogitsLoss')
axes[0].set_title('Curva de pérdida')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_f1s, label='Train F1', color='steelblue')
axes[1].plot(val_f1s,   label='Val F1',   color='darkorange')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('Curva de F1-Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=120, bbox_inches='tight')
plt.show()
print("Curvas de entrenamiento guardadas.")

## Evaluación del modelo

In [ ]:
# ─────────────────────────────────────────────────────────
# Evaluación sobre el conjunto de TEST (nunca visto durante entrenamiento)
# ─────────────────────────────────────────────────────────
model.eval()
all_preds_test  = []
all_probs_test  = []
all_labels_test = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch).squeeze(1)
        probs  = torch.sigmoid(logits)
        preds  = (probs >= 0.5).long()
        all_preds_test.extend(preds.numpy())
        all_probs_test.extend(probs.numpy())
        all_labels_test.extend(y_batch.numpy())

all_preds_test  = np.array(all_preds_test)
all_probs_test  = np.array(all_probs_test)
all_labels_test = np.array(all_labels_test, dtype=int)

# ─────────────────────────────────────────────────────────
# Métricas
# Justificación de selección de métricas:
# - Accuracy: útil como referencia pero engañosa con clases desbalanceadas
#   (un modelo que siempre predice "no legendario" obtiene ~91% de accuracy).
# - F1-Score: media armónica de precision y recall, penaliza los errores en
#   la clase minoritaria. Es la métrica más importante para este problema.
# - AUC-ROC: mide la capacidad discriminativa del modelo a distintos umbrales.
#   Un valor de 1.0 es perfecto; 0.5 es aleatorio.
# - Matriz de confusión: permite ver los errores de forma detallada.
# ─────────────────────────────────────────────────────────
acc  = accuracy_score(all_labels_test, all_preds_test)
f1   = f1_score(all_labels_test, all_preds_test, zero_division=0)
auc  = roc_auc_score(all_labels_test, all_probs_test)

print("=" * 50)
print("MÉTRICAS EN CONJUNTO DE TEST")
print("=" * 50)
print(f"Accuracy : {acc:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")
print()
print("Reporte completo:")
print(classification_report(all_labels_test, all_preds_test,
                             target_names=['No Legendario', 'Legendario']))

In [ ]:
# ─────────────────────────────────────────────────────────
# Matriz de confusión
# ─────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels_test, all_preds_test)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No Leg.', 'Pred: Legendario'],
            yticklabels=['Real: No Leg.', 'Real: Legendario'],
            ax=ax)
ax.set_title('Matriz de Confusión — Conjunto de Test')
ax.set_ylabel('Etiqueta Real')
ax.set_xlabel('Etiqueta Predicha')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

TN, FP, FN, TP = cm.ravel()
print(f"VN (True Negative)  = {TN}")
print(f"FP (False Positive) = {FP}")
print(f"FN (False Negative) = {FN}")
print(f"VP (True Positive)  = {TP}")

## Preguntas finales

### 1. Interpretación de la matriz de confusión

**Definición de cada tipo de error:**

- **Verdadero Positivo (VP / TP)**: El modelo predice "Legendario" y el Pokémon **sí** es legendario. ✅
- **Verdadero Negativo (VN / TN)**: El modelo predice "No legendario" y el Pokémon **no** es legendario. ✅
- **Falso Positivo (FP)**: El modelo predice "Legendario" pero el Pokémon **no** es legendario. ❌ (Error Tipo I)
- **Falso Negativo (FN)**: El modelo predice "No legendario" pero el Pokémon **sí** es legendario. ❌ (Error Tipo II)

**¿Elegiría Pokémon en FP o FN para mi equipo?**

- Los **Falsos Positivos (FP)** son Pokémon comunes que el modelo creyó que eran legendarios. En la práctica, esto podría ser interesante: el modelo detectó que ese Pokémon tiene estadísticas inusualmente altas para ser "común". Un ejemplo típico serían pseudo-legendarios como Dragonite o Tyranitar. Para un equipo competitivo, podría ser una buena elección ya que son fuertes.

- Los **Falsos Negativos (FN)** son Pokémon legendarios que el modelo clasificó como comunes. Esto podría indicar que son legendarios "atípicos" con estadísticas bajas (como Cosmog o Phione). Para un equipo de combate, no sería necesariamente la mejor opción.

**Conclusión**: Elegiría los **FP** por sobre los **FN**, ya que el modelo los confundió con legendarios por sus estadísticas sobresalientes, lo que los hace candidatos competitivos valiosos.

### 2. Caso mal clasificado: Falso Negativo

In [ ]:
# Encontrar casos mal clasificados (Falsos Negativos: legendarios predichos como no-legendarios)
# Reconstruimos el dataframe con índices del test set para identificar los Pokémon

# Obtener los índices del test set reproduciendo el split
_, idx_test = train_test_split(
    np.arange(len(data)), test_size=0.15, stratify=data['Legendary'].astype(int), random_state=42)

test_df = data.iloc[idx_test].copy().reset_index(drop=True)
test_df['Predicted'] = all_preds_test
test_df['Probability'] = all_probs_test.round(3)
test_df['Correct'] = (test_df['Legendary'].astype(int) == test_df['Predicted'])

# Falsos Negativos: legendarios clasificados como no-legendarios
false_negatives = test_df[(test_df['Legendary'] == True) & (test_df['Predicted'] == 0)]
print("Falsos Negativos (Legendarios clasificados como No-Legendarios):")
print(false_negatives[['Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack',
                         'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Probability']].to_string())

**Interpretación del caso mal clasificado:**

Un ejemplo típico de Falso Negativo podría ser **Cosmog** (o un Pokémon legendario con estadísticas bajas). Cosmog tiene un Total de solo 200 puntos, muy inferior al promedio de los legendarios (alrededor de 600). El modelo aprendió que los legendarios tienen estadísticas altas, por lo que un legendario con estadísticas comparables a un Pokémon común es difícil de clasificar correctamente.

Esto ilustra una limitación del modelo: confunde la "condición legendaria" con "estadísticas altas", cuando en realidad hay legendarios en etapas previas de evolución (pre-legendarios) que tienen estadísticas bajas. El modelo aprendió un atajo estadístico en lugar de capturar el concepto completo.

### 3. Mayor desafío

El mayor desafío fue **el desbalance de clases** (735 vs 65 instancias legendarias, aprox. 8% de la clase positiva).

**¿Cómo se resolvió?**

1. **pos_weight en BCEWithLogitsLoss**: Se calculó el peso inverso de frecuencias (~11.3) para penalizar más los errores en la clase minoritaria durante el entrenamiento.
2. **Particiones estratificadas**: Se usó `stratify=y` en `train_test_split` para garantizar que la proporción de legendarios se mantuviera en todos los splits.
3. **Métricas adecuadas**: Se priorizó el F1-Score y AUC-ROC sobre el Accuracy, ya que un modelo trivial que siempre predice "No legendario" obtendría ~91% de accuracy sin aprender nada útil.
4. **Dropout**: La regularización por Dropout evitó que el modelo simplemente memorizara los pocos ejemplos legendarios del set de entrenamiento.

## IA Generativa

1. **¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea?**

   Sí. Se utilizó **Claude (Anthropic)** como herramienta de apoyo.

2. **¿En qué parte o partes de la tarea utilizó estas herramientas?**

   - Estructura general del notebook y del archivo `models.py`.
   - Justificaciones de las decisiones de preprocesamiento, arquitectura, optimizador y métricas.
   - Código de entrenamiento con early stopping y checkpointing.
   - Respuestas a las preguntas finales como punto de partida para la reflexión propia.